In [2]:
import os
import re
import math
import csv
from collections import Counter

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-ż ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return [c for c in text if c != ' ']

def build_profile(folder):
    profile = Counter()
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if os.path.isfile(path):
            with open(path, 'r', encoding='utf-8') as f:
                text = f.read()
                profile.update(preprocess(text))
    total = sum(profile.values())
    for k in profile:
        profile[k] /= total
    return profile

# 
def euclidean(p1, p2):
    all_keys = set(p1.keys()).union(p2.keys())
    return math.sqrt(sum((p1.get(k,0)-p2.get(k,0))**2 for k in all_keys))

def manhattan(p1, p2):
    all_keys = set(p1.keys()).union(p2.keys())
    return sum(abs(p1.get(k,0)-p2.get(k,0)) for k in all_keys)

def maximum(p1, p2):
    all_keys = set(p1.keys()).union(p2.keys())
    return max(abs(p1.get(k,0)-p2.get(k,0)) for k in all_keys)

def cosine(p1, p2):
    all_keys = set(p1.keys()).union(p2.keys())
    dot = sum(p1.get(k,0)*p2.get(k,0) for k in all_keys)
    norm1 = math.sqrt(sum(v**2 for v in p1.values()))
    norm2 = math.sqrt(sum(v**2 for v in p2.values()))
    return dot / (norm1*norm2) if norm1*norm2 != 0 else 0

korpusy_folder = "./KORPUSY"
languages = [d for d in os.listdir(korpusy_folder) if os.path.isdir(os.path.join(korpusy_folder, d))]
profiles = {lang: build_profile(os.path.join(korpusy_folder, lang)) for lang in languages}

with open("./KORPUSY/sample2.txt", "r", encoding="utf-8") as f:
    sample_text = f.read()
sample_profile = Counter(preprocess(sample_text))
total = sum(sample_profile.values())
for k in sample_profile:
    sample_profile[k] /= total

results = {}
for lang, prof in profiles.items():
    results[lang] = {
        'euklides': euclidean(sample_profile, prof),
        'taksowkowa': manhattan(sample_profile, prof),
        'maksimum': maximum(sample_profile, prof),
        'kosinus': cosine(sample_profile, prof)
    }

with open("language_detection_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Język", "Euklides", "Taksówkowa", "Maksimum", "Kosinus"])
    for lang, metrics in results.items():
        writer.writerow([lang, f"{metrics['euklides']:.4f}", f"{metrics['taksowkowa']:.4f}",
                         f"{metrics['maksimum']:.4f}", f"{metrics['kosinus']:.4f}"])

predicted = min(results.items(), key=lambda x: x[1]['euklides'])[0]
with open("language_detection_results.csv", "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([])
    writer.writerow(["Przewidywany język", predicted])
